## Autocomplete phrases based on loaded documents

In [1]:
#%pip install --upgrade --quiet google-genai

In [2]:
MODEL_ID = "models/gemini-1.5-flash-002"

import os
from dotenv import load_dotenv
load_dotenv("../keys.env");

In [3]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

## Load documents into context (once)
Use context caching so that we do not need to use input tokens for each call.
This is particularly useful if we are providing functionality based on video.

In [4]:
# Download text of play from Project Gutenberg
TXT_URL="https://www.gutenberg.org/cache/epub/1522/pg1522.txt"
LOCAL_FILE="julius_caesar.txt"

import requests

def download_text_file(url, file_path):
    response = requests.get(url)
    response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
    with open(file_path, "wb") as file:
        file.write(response.content)
    print(f"File downloaded successfully to {file_path}")

download_text_file(TXT_URL, LOCAL_FILE)

File downloaded successfully to julius_caesar.txt


In [5]:
# upload to Gemini
uploaded_file = client.files.upload(file=LOCAL_FILE)
uploaded_file

File(name='files/kjfmomvvxcgd', display_name=None, mime_type='text/plain', size_bytes=142275, create_time=datetime.datetime(2025, 2, 17, 3, 34, 10, 862913, tzinfo=TzInfo(UTC)), expiration_time=datetime.datetime(2025, 2, 19, 3, 34, 10, 852396, tzinfo=TzInfo(UTC)), update_time=datetime.datetime(2025, 2, 17, 3, 34, 10, 862913, tzinfo=TzInfo(UTC)), sha256_hash='MGQ4NzQzYmU1YTkwMTdlMmE3NzFiZjAzNGI5OTBiZWY2OWFkYjQzNDNhZWM0Y2U4M2NlODI4NzI4YmFlOTZkMA==', uri='https://generativelanguage.googleapis.com/v1beta/files/kjfmomvvxcgd', download_uri=None, state=<FileState.ACTIVE: 'ACTIVE'>, source=<FileSource.UPLOADED: 'UPLOADED'>, video_metadata=None, error=None)

In [6]:
context_cache = client.caches.create(
    model=MODEL_ID,
    config=types.CreateCachedContentConfig(
        contents=[uploaded_file],
        system_instruction="""
        Using the provided document, provide the best way to complete the phrase.
        Return an empty string if no match is found.
        Return only the completion, without any preamble.
        """,
    )
)

context_cache

CachedContent(name='cachedContents/us51vfbflur6', display_name='', model='models/gemini-1.5-flash-002', create_time=datetime.datetime(2025, 2, 17, 3, 34, 11, 526415, tzinfo=TzInfo(UTC)), update_time=datetime.datetime(2025, 2, 17, 3, 34, 11, 526415, tzinfo=TzInfo(UTC)), expire_time=datetime.datetime(2025, 2, 17, 4, 34, 11, 351987, tzinfo=TzInfo(UTC)), usage_metadata=CachedContentUsageMetadata(audio_duration_seconds=None, image_count=None, text_count=None, total_token_count=40967, video_duration_seconds=None))

## Now use the cached content to generate autocomplete suggestions

These will be grounded on the document, rather than on previous users' queries

In [7]:
# Assume the user has typed this in
typed_text = "Friends should"
response = client.models.generate_content(
    model=MODEL_ID,
    contents=typed_text,
    config=types.GenerateContentConfig(
        temperature=0.9,
        cached_content=context_cache.name,
        candidate_count=3,
        safety_settings=[
            types.SafetySetting(
                category='HARM_CATEGORY_HATE_SPEECH',
                threshold='BLOCK_ONLY_HIGH',
            )
        ]
    )
)

In [8]:
print([(cand.content.parts[0].text, cand.avg_logprobs) for cand in response.candidates])

[('bear his friend’s infirmities\n', -0.06801926344633102), ('bear his friend’s infirmities\n', -0.06813424080610275), ('bear his friend’s infirmities\n', -0.07367707043886185)]
